# TrialSpec-LM: Kaggle training runner

Thin wrapper: all logic lives in the GitHub repo (`src/`). This notebook clones it and runs the scripts.

**Settings:** Accelerator = GPU T4 x2, Internet = On, Secrets attached: `WANDB_API_KEY`, `HF_TOKEN`.

**Flow (one Save & Run All):** tests → smoke train → smoke predict → *(only if all pass)* zero-shot baseline on val → full fine-tune → fine-tuned predictions on val → zip results.
Any failing step raises an error and stops the notebook, so a broken setup never burns a long run.

In [ ]:
RUN_FULL_AFTER_SMOKE = True   # False = stop after the smoke tests
RUN_BASELINE = True           # zero-shot 'before fine-tuning' predictions on val
BRANCH = 'master'
CONFIG = 'configs/train.yaml'
ADAPTER_DIR = 'outputs/qwen2.5-1.5b-lora/final'

In [ ]:
import os, subprocess, sys, time

def run(cmd):
    """Run a shell command, stream its output, and raise if it fails (stops Save & Run All)."""
    print(f'\n$ {cmd}', flush=True)
    t0 = time.time()
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    if p.wait() != 0:
        raise RuntimeError(f'FAILED (exit {p.returncode}): {cmd}')
    print(f'[ok in {time.time() - t0:.0f}s]', flush=True)

REPO = '/kaggle/working/TrialSpec-LM'
if os.path.exists(REPO):
    run(f'git -C {REPO} pull')
else:
    run(f'git clone -b {BRANCH} https://github.com/Umairishuman/TrialSpec-LM.git {REPO}')
os.chdir(REPO)
run('git log --oneline -1')
run('bash scripts/kaggle_setup.sh')   # environment fixes live in the repo

In [ ]:
from kaggle_secrets import UserSecretsClient
s = UserSecretsClient()
os.environ['WANDB_API_KEY'] = s.get_secret('WANDB_API_KEY')
os.environ['HF_TOKEN'] = s.get_secret('HF_TOKEN')
os.environ['CUDA_VISIBLE_DEVICES'] = '0'   # single GPU (decisions log D-010)
os.environ['PYTHONUNBUFFERED'] = '1'

import torch, transformers, peft, numpy, pandas, datasets
print('torch', torch.__version__, '| transformers', transformers.__version__, '| peft', peft.__version__,
      '| numpy', numpy.__version__, '| pandas', pandas.__version__, '| datasets', datasets.__version__)
if not torch.cuda.is_available():
    raise RuntimeError(f'No GPU (torch {torch.__version__}). Set Session options -> Accelerator = GPU T4 x2, then Save Version again.')
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), 'GiB')

## 1. Tests + smoke runs (a few minutes). Any failure stops here.

In [ ]:
run('python -m pytest -q tests')
run('python -m src.train.run --smoke')
run('python -m src.eval.predict --smoke --split val --out results/smoke/baseline_val')
print('\n===== ALL SMOKE TESTS PASSED =====')

## 2. Zero-shot baseline (before fine-tuning) on val

In [ ]:
if RUN_FULL_AFTER_SMOKE and RUN_BASELINE:
    run('python -m src.eval.predict --split val --out results/baseline/qwen2.5-1.5b_val')

## 3. Full fine-tune, then fine-tuned predictions on val (same examples, same greedy decoding)

In [ ]:
if RUN_FULL_AFTER_SMOKE:
    run(f'python -m src.train.run --config {CONFIG}')
    run(f'python -m src.eval.predict --split val --adapter {ADAPTER_DIR} --out results/finetuned/qwen2.5-1.5b-lora_val')
    print('\n===== FULL RUN FINISHED =====')

In [ ]:
# /kaggle/working is kept as the version's output. Download results.zip from the Output tab.
run('cd /kaggle/working && rm -f results.zip && zip -qr results.zip TrialSpec-LM/results TrialSpec-LM/outputs/*/final && ls -la /kaggle/working')